# Analysis based on DL-Hard Dataset

This part is based on the dataset called DL-Hard, which is an annotated dataset buiilt upon TREC Deep Learning questions. The questions are annotated with query intent categories based on the paper An Intent Taxonomy for Questions Asked in Web Search (https://marksanderson.org/files/papers/CHIIR21b.pdf) and answer type which is a manual annotation of target answer type for we questions (short answer, factoid etc). They also identified 50 hard questions from the ~100 TREC DL 2019/2020 dataset, which we use to compare QE technique performance on hard and non-hard queries. 

## Grouping queries based on query intent and answer type

In [1]:
# This cell loads the annotated TREC DL queries and the common qrels used for the shared evaluation setting.
import pyterrier as pt
import pandas as pd
dataset = pt.get_dataset("msmarco_passage")
index = dataset.get_index(variant="terrier_stemmed")

# Load DL-hard annotations
annotations_url = "https://raw.githubusercontent.com/grill-lab/DL-Hard/main/annotations/query/annotations.tsv"
annotations = pd.read_csv(
    annotations_url,
    sep="\t",
    header=None,
    names=["qid", "query", "intent", "answer_type", "topic_domain", "serp_type"],
)
annotations["qid"] = annotations["qid"].astype(str)

# Load TREC DL 2019/2020 qrels. This dataset contains relevance judgements from scale 0 to 3.
dl19 = pt.get_dataset("irds:msmarco-passage/trec-dl-2019")
dl20 = pt.get_dataset("irds:msmarco-passage/trec-dl-2020")
all_qrels = pd.concat([dl19.get_qrels(), dl20.get_qrels()])
all_qrels["qid"] = all_qrels["qid"].astype(str)

# Keep only queries that have both annotations and judgements.
qrels_qids = set(all_qrels["qid"].unique())
topics = annotations[annotations["qid"].isin(qrels_qids)].reset_index(drop=True)
print(f"Queries with both annotations and relevance values: {len(topics)}")
print(topics["intent"].value_counts())
print("\n")
print(topics["answer_type"].value_counts())


Queries with both annotations and relevance values: 97
intent
description     48
list            10
quantity         9
reason           8
attribute        7
entity           5
verification     3
process          3
language         3
weather          1
Name: count, dtype: int64


answer_type
factoid              28
definition           21
long answer          16
list                 10
short description     7
comparison            5
short answer          5
guide                 2
weather               1
Long answer           1
multi-answer          1
Name: count, dtype: int64


In [2]:
# This cell collapses the fine-grained intent and answer-type labels into broader analysis groups.
# Answer type can be further divided instead of keeping 12 categories
intent_map = {
    "description": "Informational",
    "reason": "Informational",
    "process": "Informational",
    "entity": "Fact-Seeking",
    "attribute": "Fact-Seeking",
    "verification": "Fact-Seeking",
    "language": "Fact-Seeking",
    "list": "Data/List",
    "quantity": "Data/List",
    "weather": "Data/List",
}

answer_map = {
    "definition": "Narrative",
    "long answer": "Narrative",
    "Long answer": "Narrative",
    "short description": "Narrative",
    "guide": "Narrative",
    "factoid": "Atomic",
    "short answer": "Atomic",
    "weather": "Atomic",
    "list": "Complex",
    "comparison": "Complex",
    "multi-answer": "Complex",
}
topics["meta_intent"] = topics["intent"].map(intent_map)
print(topics["meta_intent"].value_counts(), "\n")
topics["meta_answer_type"] = topics["answer_type"].map(answer_map)
print(topics["meta_answer_type"].value_counts())


meta_intent
Informational    59
Data/List        20
Fact-Seeking     18
Name: count, dtype: int64 

meta_answer_type
Narrative    47
Atomic       34
Complex      16
Name: count, dtype: int64


## Classical QE techniques

In [3]:
# This cell cleans the queries and initializes the BM25, RM3, and Bo1 retrieval pipelines.
import re

def clean_query(q):
    q = re.sub(r"[^\w\s]", " ", str(q))
    q = re.sub(r"\s+", " ", q).strip()
    return q

# Clean queries (else it gives errors in PyTerrier)
topics["query"] = topics["query"].apply(clean_query)

# Initialize BM25 retriever
bm25 = pt.terrier.Retriever(index, wmodel="BM25")

# Initialize classic query expansion models
rm3 = pt.rewrite.RM3(index, fb_terms=10, fb_docs=3)
bo1 = pt.rewrite.Bo1QueryExpansion(index, fb_terms=10, fb_docs=3)

bm25_rm3 = bm25 >> rm3 >> bm25
bm25_bo1 = bm25 >> bo1 >> bm25


Java started (triggered by Retriever.__init__) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


## LLM-based QE techniques 

In [4]:
# This cell installs the local Ollama client if it is not already available in the notebook environment.
import sys
!{sys.executable} -m pip install ollama


'd:\MSc' is not recognized as an internal or external command,
operable program or batch file.


## Query2doc

In [5]:
# This cell loads cached Query2Doc expansions when available, otherwise it generates and saves them.
import ollama
import pandas as pd
from tqdm import tqdm
import os


file_path = "query2doc/topics_with_q2d.csv"

if os.path.exists(file_path):
    q2d_topics = pd.read_csv(file_path)
    q2d_topics["qid"] = q2d_topics["qid"].astype(str)
    q2d_topics["q2d_query"] = q2d_topics["q2d_query"].apply(clean_query)

else:
    print("Generating query2doc documents...")

    def expand_query_sparse(original_query, llm_generated_doc):
        # Paper recommends boosting the original query by appending it 5 times, followed by the generated pseudo-document.
        boosted_query = (original_query + " ") * 5
        return boosted_query + llm_generated_doc

    def generate_q2d_query(query):

        system_msg = "You are a specialized query expansion assistant. Your ONLY task is to write a single informative passage that answers the user's target query. Do not provide multiple answers or refer to previous examples."
        # Generate a pseudo-document using the original query as input to the LLM.
        # We use a few-shot prompt to guide the LLM in generating a passage that is relevant to the query.
        # The examples are directly taken from the paper.
        prompt = f"""### Instruction ### Write a passage that answers the given query:

        ### Examples ###
        Query: what state is this zip code 85282
        Passage: Welcome to TEMPE, AZ 85282. 85282 is a rural zip code in Tempe, Arizona. The population
        is primarily white, and mostly single. At $200,200 the average home value here is a bit higher than
        average for the Phoenix-Mesa-Scottsdale metro area, so this probably isn’t the place to look for housing
        bargains.5282 Zip code is located in the Mountain time zone at 33 degrees latitude (Fun Fact: this is the
        same latitude as Damascus, Syria!) and -112 degrees longitude.

        Query: why is gibbs model of reflection good
        Passage: In this reflection, I am going to use Gibbs (1988) Reflective Cycle. This model is a recognised
        framework for my reflection. Gibbs (1988) consists of six stages to complete one cycle which is able
        to improve my nursing practice continuously and learning from the experience for better practice in the
        future.n conclusion of my reflective assignment, I mention the model that I chose, Gibbs (1988) Reflective
        Cycle as my framework of my reflective. I state the reasons why I am choosing the model as well as some
        discussion on the important of doing reflection in nursing practice.

        Query: what does a thousand pardons means
        Passage: Oh, that’s all right, that’s all right, give us a rest; never mind about the direction, hang the
        direction - I beg pardon, I beg a thousand pardons, I am not well to-day; pay no attention when I soliloquize,
        it is an old habit, an old, bad habit, and hard to get rid of when one’s digestion is all disordered with eating
        food that was raised forever and ever before he was born; good land! a man can’t keep his functions
        regular on spring chickens thirteen hundred years old.

        Query: what is a macro warning
        Passage: Macro virus warning appears when no macros exist in the file in Word. When you open
        a Microsoft Word 2002 document or template, you may receive the following macro virus warning,
        even though the document or template does not contain macros: C:\<path>\<file name>contains macros.
        Macros may contain viruses.

        ### Task ###
        Query: {query}
        Passage:"""

        response = ollama.generate(
            model="llama3.1",
            prompt=prompt,
            system=system_msg,
            options={
                "stop": [
                    "Query:",
                    "###",
                ],  
                "num_predict": 150,  
            },
        )

        return expand_query_sparse(query, response["response"])

    # Since local generation takes time, use a progress bar
    tqdm.pandas()
    topics["q2d_query"] = topics["query"].progress_apply(generate_q2d_query)
    topics["q2d_query"] = (
        topics["q2d_query"].str.replace(r"\s+", " ", regex=True).str.strip()
    )

    q2d_topics = topics[["qid", "q2d_query"]]

    q2d_topics.to_csv("query2doc/topics_with_q2d.csv", index=False)
    print(f"Q2D queries generated and saved to {file_path}")


<>:90: SyntaxWarning: invalid escape sequence '\<'
<>:90: SyntaxWarning: invalid escape sequence '\<'
C:\Users\radha\AppData\Local\Temp\ipykernel_4396\3023634566.py:90: SyntaxWarning: invalid escape sequence '\<'


## Chain-of-thoughts

In [6]:
# This cell loads cached Chain-of-Thought expansions when available, otherwise it generates and saves them.
file_path = "cot/topics_with_cot.csv"

if os.path.exists(file_path):
    cot_topics = pd.read_csv(file_path)
    cot_topics["qid"] = cot_topics["qid"].astype(str)
    cot_topics["cot_query"] = cot_topics["cot_query"].apply(clean_query)
    print(f"Loading existing CoT expansions from {file_path}...")
else:
    print("Generating Chain-of-Thought expansions...")

    def generate_cot_query(query):

        system_msg = (
            "You are a retrieval assistant. Given a query, write a short factual passage "
            "with supporting context, then end with 'So the final answer is [answer].' "
            "Be concise. Do not use bullet points or step-by-step reasoning."
        )

        prompt = f"""Answer the following queries with a short factual passage followed by a final answer sentence.

        Query: who owns jaguar land rover?
        Passage: Jaguar Land Rover is a British multinational car manufacturer founded by William Lyons in 1931. Its headquarters are in Whitley, Coventry, United Kingdom. The company is a wholly owned subsidiary of Tata Motors of India. So the final answer is Tata Motors.

        Query: what year was the eiffel tower built?
        Passage: The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars in Paris, France. It was designed by Gustave Eiffel and constructed between 1887 and 1889 as the entrance arch for the 1889 World's Fair. So the final answer is 1889.

        Query: what is the capital of australia?
        Passage: Australia is a federal parliamentary constitutional monarchy. Its capital city is Canberra, which was purpose-built as a compromise between rivals Sydney and Melbourne. Canberra became the capital in 1913. So the final answer is Canberra.

        Query: {query}
        Passage:"""

        response = ollama.generate(
            model="llama3.1",
            prompt=prompt,
            system=system_msg,
            options={
                "num_predict": 150,
            },
        )

        return query + response["response"]

    topics["cot_query"] = topics["query"].progress_apply(generate_cot_query)
    topics["cot_query"] = topics["cot_query"].apply(clean_query)

    cot_topics = topics[["qid", "cot_query"]]
    cot_topics.to_csv(file_path, index=False)
    print(f"CoT queries generated and saved to {file_path}")


Loading existing CoT expansions from cot/topics_with_cot.csv...


# Running the experiments

In [7]:
# This cell runs all retrieval experiments on the common TREC DL qrels,
# then builds Query2Doc-vs-traditional per-query delta tables for the main subgroup analyses.
from pyterrier.measures import RR, nDCG, AP, R

target_method = "Q2D_Llama3"
traditional_methods = ["BM25", "RM3", "Bo1"]
metric_order = ["nDCG@10", "AP(rel=2)", "RR(rel=2)@10", "R(rel=2)@100"]

if "q2d_query" in q2d_topics.columns:
    q2d_topics = q2d_topics.rename(columns={"q2d_query": "query"})

results_classical = pt.Experiment(
    [bm25, bm25_rm3, bm25_bo1],
    topics[["qid", "query"]],
    all_qrels,
    eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
    names=["BM25", "RM3", "Bo1"],
    perquery=True,
)

results_q2d = pt.Experiment(
    [bm25],
    q2d_topics[["qid", "query"]],
    all_qrels,
    eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
    names=[target_method],
    perquery=True,
)

results_cot = pt.Experiment(
    [bm25],
    cot_topics[["qid", "cot_query"]].rename(columns={"cot_query": "query"}),
    all_qrels,
    eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
    names=["CoT_Llama3"],
    perquery=True,
)

results = pd.concat([results_classical, results_q2d, results_cot], ignore_index=True)
label_map = topics[["qid", "query", "meta_intent", "meta_answer_type"]]
results = results.merge(label_map, on="qid", how="left")

def build_delta_frame(results_df, target_name, baseline_names, metadata_cols):
    delta_rows = []
    for measure in metric_order:
        measure_df = results_df[results_df["measure"] == measure]
        score_pivot = measure_df.pivot(index="qid", columns="name", values="value")
        meta_df = measure_df[["qid"] + metadata_cols].drop_duplicates("qid")
        merged = meta_df.merge(score_pivot.reset_index(), on="qid", how="left")
        if target_name not in merged.columns:
            continue

        for baseline_name in baseline_names:
            if baseline_name not in merged.columns:
                continue
            paired = merged[["qid"] + metadata_cols + [target_name, baseline_name]].dropna().copy()
            paired["comparison"] = f"{target_name} - {baseline_name}"
            paired["measure"] = measure
            paired["target_score"] = paired[target_name]
            paired["baseline_score"] = paired[baseline_name]
            paired["delta"] = paired["target_score"] - paired["baseline_score"]
            delta_rows.append(
                paired[
                    [
                        "qid",
                        *metadata_cols,
                        "comparison",
                        "measure",
                        "target_score",
                        "baseline_score",
                        "delta",
                    ]
                ]
            )

    if not delta_rows:
        return pd.DataFrame(
            columns=[
                "qid",
                *metadata_cols,
                "comparison",
                "measure",
                "target_score",
                "baseline_score",
                "delta",
            ]
        )

    return pd.concat(delta_rows, ignore_index=True)

delta_results = build_delta_frame(
    results,
    target_method,
    traditional_methods,
    ["query", "meta_intent", "meta_answer_type"],
)

print("\n=== Overall Results ===")
overall = results.groupby(["name", "measure"])["value"].mean().unstack()
print(overall)

print("\n=== nDCG@10 by Query Intent ===")
ndcg_by_intent = (
    results[results["measure"] == "nDCG@10"]
    .groupby(["name", "meta_intent"])["value"]
    .mean()
    .unstack()
)
print(ndcg_by_intent)

print("\n=== nDCG@10 by Answer Type ===")
ndcg_by_answer_type = (
    results[results["measure"] == "nDCG@10"]
    .groupby(["name", "meta_answer_type"])["value"]
    .mean()
    .unstack()
)
print(ndcg_by_answer_type)

print("\n=== Mean Query2Doc Gain by Query Intent (nDCG@10) ===")
intent_gain_summary = (
    delta_results[delta_results["measure"] == "nDCG@10"]
    .groupby(["comparison", "meta_intent"])["delta"]
    .mean()
    .unstack()
)
print(intent_gain_summary)

print("\n=== Mean Query2Doc Gain by Answer Type (nDCG@10) ===")
answer_gain_summary = (
    delta_results[delta_results["measure"] == "nDCG@10"]
    .groupby(["comparison", "meta_answer_type"])["delta"]
    .mean()
    .unstack()
)
print(answer_gain_summary)



=== Overall Results ===
measure     AP(rel=2)  R(rel=2)@100  RR(rel=2)@10   nDCG@10
name                                                       
BM25         0.290089      0.541421      0.625749  0.487382
Bo1          0.311664      0.568762      0.618295  0.500851
CoT_Llama3   0.338962      0.559121      0.643941  0.514828
Q2D_Llama3   0.389330      0.628159      0.756554  0.578139
RM3          0.317579      0.575153      0.628371  0.516954

=== nDCG@10 by Query Intent ===
meta_intent  Data/List  Fact-Seeking  Informational
name                                               
BM25          0.462645      0.496273       0.493056
Bo1           0.516190      0.494730       0.497519
CoT_Llama3    0.505907      0.481448       0.528036
Q2D_Llama3    0.560193      0.573080       0.585766
RM3           0.505432      0.514619       0.521572

=== nDCG@10 by Answer Type ===
meta_answer_type    Atomic   Complex  Narrative
name                                           
BM25              0.510066  0.

In [8]:
# This cell installs matplotlib for any optional plotting used later in the notebook.
!pip install matplotlib 



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# This cell tests whether Query2Doc beats BM25 and traditional QE overall,
# then lists representative queries where the best traditional QE method still beats Query2Doc.
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

comparison_order = [f"{target_method} - {baseline}" for baseline in traditional_methods]

def apply_holm(df, p_col="p_value"):
    df = df.copy()
    if len(df) == 0:
        df["p_corrected"] = pd.Series(dtype=float)
        df["significant"] = pd.Series(dtype=bool)
        return df
    reject, p_corr, _, _ = multipletests(df[p_col], method="holm")
    df["p_corrected"] = p_corr
    df["significant"] = reject
    return df

def run_overall_paired_tests(results_df, target_name, baseline_names):
    rows = []
    for measure in metric_order:
        measure_df = results_df[results_df["measure"] == measure]
        score_pivot = measure_df.pivot(index="qid", columns="name", values="value")
        if target_name not in score_pivot.columns:
            continue

        for baseline_name in baseline_names:
            if baseline_name not in score_pivot.columns:
                continue
            paired = score_pivot[[target_name, baseline_name]].dropna()
            if len(paired) < 2:
                continue

            diffs = paired[target_name] - paired[baseline_name]
            if (diffs == 0).all():
                stat, p = 0.0, 1.0
            else:
                stat, p = wilcoxon(paired[target_name], paired[baseline_name], alternative="two-sided")

            rows.append(
                {
                    "comparison": f"{target_name} - {baseline_name}",
                    "measure": measure,
                    "n_pairs": len(paired),
                    "target_mean": round(float(paired[target_name].mean()), 4),
                    "baseline_mean": round(float(paired[baseline_name].mean()), 4),
                    "mean_delta": round(float(diffs.mean()), 4),
                    "median_delta": round(float(diffs.median()), 4),
                    "wilcoxon_stat": round(float(stat), 4),
                    "p_value": float(p),
                }
            )

    overall_df = apply_holm(pd.DataFrame(rows))
    if len(overall_df) > 0:
        overall_df["comparison"] = pd.Categorical(overall_df["comparison"], categories=comparison_order, ordered=True)
        overall_df["measure"] = pd.Categorical(overall_df["measure"], categories=metric_order, ordered=True)
    return overall_df

overall_delta_tests = run_overall_paired_tests(results, target_method, traditional_methods)
print("=== Overall paired Query2Doc vs baseline comparisons (Holm-corrected) ===")
print(overall_delta_tests.sort_values(["measure", "p_corrected"]).to_string(index=False))

metric = "nDCG@10"
per_query = (
    results[results["measure"] == metric]
    .pivot_table(index="qid", columns="name", values="value")
    .reset_index()
)
per_query = per_query.merge(
    topics[["qid", "query", "meta_intent", "meta_answer_type"]],
    on="qid",
    how="left",
)

per_query["best_traditional"] = per_query[["RM3", "Bo1"]].max(axis=1)
per_query["best_traditional_name"] = per_query[["RM3", "Bo1"]].idxmax(axis=1)
per_query["traditional_advantage"] = per_query["best_traditional"] - per_query[target_method]

traditional_wins = (
    per_query[per_query["best_traditional"] > per_query[target_method]]
    .sort_values("traditional_advantage", ascending=False)
    .head(10)[
        [
            "query",
            "best_traditional_name",
            "best_traditional",
            target_method,
            "traditional_advantage",
            "meta_intent",
            "meta_answer_type",
        ]
    ]
    .rename(
        columns={
            "best_traditional_name": "traditional_method",
            "best_traditional": "traditional_nDCG@10",
            target_method: "query2doc_nDCG@10",
            "traditional_advantage": "advantage",
        }
    )
    .reset_index(drop=True)
)

print("\nTop queries where the best traditional QE method outperformed Query2Doc (nDCG@10):\n")
print(
    f"Total: {len(per_query[per_query['best_traditional'] > per_query[target_method]])} out of {len(per_query)} queries\n"
)
display(
    traditional_wins.style.format(
        {
            "traditional_nDCG@10": "{:.3f}",
            "query2doc_nDCG@10": "{:.3f}",
            "advantage": "{:.3f}",
        }
    )
)


=== Overall paired Query2Doc vs baseline comparisons (Holm-corrected) ===
       comparison      measure  n_pairs  target_mean  baseline_mean  mean_delta  median_delta  wilcoxon_stat      p_value  p_corrected  significant
Q2D_Llama3 - BM25      nDCG@10       97       0.5781         0.4874      0.0908        0.0241          609.0 7.613743e-05 6.090994e-04         True
 Q2D_Llama3 - RM3      nDCG@10       97       0.5781         0.5170      0.0612        0.0289         1352.0 5.134446e-03 6.934447e-03         True
 Q2D_Llama3 - Bo1      nDCG@10       97       0.5781         0.5009      0.0773        0.0361         1366.0 1.689531e-03 6.934447e-03         True
Q2D_Llama3 - BM25    AP(rel=2)       97       0.3893         0.2901      0.0992        0.0344          397.0 3.408219e-08 4.089862e-07         True
 Q2D_Llama3 - Bo1    AP(rel=2)       97       0.3893         0.3117      0.0777        0.0372         1105.0 7.850457e-06 7.850457e-05         True
 Q2D_Llama3 - RM3    AP(rel=2)       9

,query,traditional_method,traditional_nDCG@10,query2doc_nDCG@10,advantage,meta_intent,meta_answer_type
0,is caffeine an narcotic,RM3,0.842,0.000,0.842,Fact-Seeking,Atomic
1,what can you do about discrimination in the workplace in oklahoma city,RM3,0.756,0.255,0.501,Informational,Narrative
2,what is the daily life of thai people,RM3,0.612,0.112,0.500,Informational,Narrative
3,dog day afternoon meaning,RM3,0.496,0.156,0.340,Informational,Narrative
4,average wedding dress alteration cost,RM3,0.856,0.558,0.299,Data/List,Atomic
5,what is wifi vs bluetooth,Bo1,0.371,0.081,0.290,Informational,Complex
6,how many sons robert kraft has,RM3,0.759,0.469,0.289,Fact-Seeking,Atomic
7,meaning of shebang,RM3,0.952,0.698,0.254,Fact-Seeking,Narrative
8,what is the most popular food in switzerland,RM3,0.870,0.630,0.240,Informational,Atomic
9,what is a statutory deed,RM3,0.944,0.713,0.232,Informational,Narrative


### Inspecting the queries 

In [10]:
# This cell prints example evaluated queries for each fine-grained answer-type label.
for answer_type in topics["answer_type"].dropna().unique():
    examples = (
        topics[topics["answer_type"] == answer_type]["query"].head(3).tolist()
    )
    count = (topics["answer_type"] == answer_type).sum()
    print(f"\n{str(answer_type).upper()} ({count} queries):")
    for ex in examples:
        print(f"  - {ex}")



SHORT ANSWER (5 queries):
  - what is an aml surveillance analyst
  - what carvedilol used for
  - what is theraderm used for

COMPARISON (5 queries):
  - difference between a company s strategy and business model is
  - difference between a mcdouble and a double cheeseburger
  - difference between rn and bsn

LIST (10 queries):
  - what types of food can you cook sous vide
  - types of dysarthria from cerebral palsy
  - causes of left ventricular hypertrophy

FACTOID (28 queries):
  - average salary for dental hygienist in nebraska
  - average annual income data analyst
  - what is reba mcentire s net worth

LONG ANSWER (16 queries):
  - how much money do motivational speakers make
  - how much would it cost to install my own wind turbine
  - causes of military suicide

SHORT DESCRIPTION (7 queries):
  - who is aziz hashim
  - why did the ancient egyptians call their land kemet or black land
  - what type of conflict does della face in o henry the gift of the magi

DEFINITION (21 que

In [11]:
# This cell prints example evaluated queries for each fine-grained intent label.
for intent_type in topics["intent"].dropna().unique():
    examples = (
        topics[topics["intent"] == intent_type]["query"].head(3).tolist()
    )
    count = (topics["intent"] == intent_type).sum()
    print(f"\n{intent_type.upper()} ({count} queries):")
    for ex in examples:
        print(f"  - {ex}")



DESCRIPTION (48 queries):
  - what is an aml surveillance analyst
  - difference between a company s strategy and business model is
  - who is aziz hashim

LIST (10 queries):
  - what types of food can you cook sous vide
  - where is the show shameless filmed
  - types of dysarthria from cerebral palsy

QUANTITY (9 queries):
  - average salary for dental hygienist in nebraska
  - how much money do motivational speakers make
  - how much would it cost to install my own wind turbine

ATTRIBUTE (7 queries):
  - what is reba mcentire s net worth
  - what type of tissue are bronchioles
  - how often to button quail lay eggs

ENTITY (5 queries):
  - who sings monk theme song
  - who killed nicholas ii of russia
  - tracheids are part of _____

REASON (8 queries):
  - causes of military suicide
  - right pelvic pain causes
  - does legionella pneumophila cause pneumonia

VERIFICATION (3 queries):
  - is caffeine an narcotic
  - do google docs auto save
  - is cdg airport in main paris

PROCE

## Analyzing queries based on query difficulty

In [12]:
# This cell loads DL-HARD topic ids only as difficulty labels and attaches them to the shared-query results,
# so the difficulty analysis stays in the same qrel setting as the rest of the notebook.
hard_topics = pd.read_csv(
    "https://raw.githubusercontent.com/grill-lab/DL-Hard/main/dataset/topics.tsv",
    sep="	",
    header=None,
    names=["qid", "query"],
)
hard_qids = set(hard_topics["qid"].astype(str))

topics = topics.drop(columns=["is_hard", "difficulty_group"], errors="ignore")
topics["is_hard"] = topics["qid"].isin(hard_qids)
topics["difficulty_group"] = topics["is_hard"].map({True: "Hard", False: "Non-Hard"})

results = results.drop(columns=["is_hard", "difficulty_group"], errors="ignore").merge(
    topics[["qid", "is_hard", "difficulty_group"]],
    on="qid",
    how="left",
)
delta_results = delta_results.drop(columns=["is_hard", "difficulty_group"], errors="ignore").merge(
    topics[["qid", "is_hard", "difficulty_group"]],
    on="qid",
    how="left",
)

print("=== Difficulty labels from DL-HARD ===")
print(topics["difficulty_group"].value_counts())


=== Difficulty labels from DL-HARD ===
difficulty_group
Non-Hard    76
Hard        21
Name: count, dtype: int64


In [13]:
# This cell summarizes raw system performance for hard and non-hard queries under the shared qrels,
# and prepares the Query2Doc delta subset used for the difficulty analysis.
difficulty_results = results[results["difficulty_group"].notna()].copy()
difficulty_delta_df = delta_results[delta_results["difficulty_group"].notna()].copy()

print("=== Results: Hard vs Non-Hard (shared TREC DL qrels) ===")
difficulty_pivot = difficulty_results.pivot_table(
    index="name",
    columns=["measure", "difficulty_group"],
    values="value",
)
print(difficulty_pivot.to_string())

print("\n=== Mean Query2Doc Gain by Difficulty ===")
difficulty_gain_summary = (
    difficulty_delta_df
    .groupby(["comparison", "measure", "difficulty_group"])["delta"]
    .mean()
    .unstack()
)
print(difficulty_gain_summary.to_string())


=== Results: Hard vs Non-Hard (shared TREC DL qrels) ===
measure          AP(rel=2)           R(rel=2)@100           RR(rel=2)@10             nDCG@10          
difficulty_group      Hard  Non-Hard         Hard  Non-Hard         Hard  Non-Hard      Hard  Non-Hard
name                                                                                                  
BM25              0.206050  0.313310     0.428960  0.572496     0.724206  0.598543  0.448907  0.498014
Bo1               0.222514  0.336298     0.422309  0.609229     0.711640  0.592502  0.438442  0.518096
CoT_Llama3        0.225974  0.370182     0.405575  0.601549     0.611111  0.653013  0.376173  0.553141
Q2D_Llama3        0.267312  0.423046     0.483685  0.668080     0.760771  0.755388  0.467670  0.608664
RM3               0.231397  0.341393     0.452663  0.608999     0.710847  0.605582  0.471034  0.529642

=== Mean Query2Doc Gain by Difficulty ===
difficulty_group                    Hard  Non-Hard
comparison        measure

In [14]:
# This cell tests whether Query2Doc's relative gains change between hard and non-hard queries.
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

comparison_order = [f"{target_method} - {baseline}" for baseline in traditional_methods]
difficulty_delta_rows = []

for comparison in comparison_order:
    for measure in metric_order:
        subset = difficulty_delta_df[
            (difficulty_delta_df["comparison"] == comparison)
            & (difficulty_delta_df["measure"] == measure)
        ]
        hard_scores = subset.loc[subset["difficulty_group"] == "Hard", "delta"].dropna().values
        nonhard_scores = subset.loc[subset["difficulty_group"] == "Non-Hard", "delta"].dropna().values
        if len(hard_scores) < 2 or len(nonhard_scores) < 2:
            continue

        stat, p = mannwhitneyu(hard_scores, nonhard_scores, alternative="two-sided")
        difficulty_delta_rows.append(
            {
                "comparison": comparison,
                "measure": measure,
                "n_hard": len(hard_scores),
                "n_nonhard": len(nonhard_scores),
                "hard_mean_delta": round(float(hard_scores.mean()), 4),
                "nonhard_mean_delta": round(float(nonhard_scores.mean()), 4),
                "delta_gap": round(float(nonhard_scores.mean() - hard_scores.mean()), 4),
                "mw_statistic": round(float(stat), 4),
                "p_value": float(p),
            }
        )

difficulty_delta_tests = pd.DataFrame(difficulty_delta_rows)
if len(difficulty_delta_tests) > 0:
    reject, p_corr, _, _ = multipletests(difficulty_delta_tests["p_value"], method="holm")
    difficulty_delta_tests["p_corrected"] = p_corr
    difficulty_delta_tests["significant"] = reject
    difficulty_delta_tests["comparison"] = pd.Categorical(
        difficulty_delta_tests["comparison"],
        categories=comparison_order,
        ordered=True,
    )
    difficulty_delta_tests["measure"] = pd.Categorical(
        difficulty_delta_tests["measure"],
        categories=metric_order,
        ordered=True,
    )

print("=== Difficulty effects on Query2Doc gains (Holm-corrected) ===")
print(difficulty_delta_tests.sort_values(["measure", "p_corrected"]).to_string(index=False))


=== Difficulty effects on Query2Doc gains (Holm-corrected) ===
       comparison      measure  n_hard  n_nonhard  hard_mean_delta  nonhard_mean_delta  delta_gap  mw_statistic  p_value  p_corrected  significant
Q2D_Llama3 - BM25      nDCG@10      21         76           0.0188              0.1106     0.0919         573.0 0.047317     0.567804        False
 Q2D_Llama3 - RM3      nDCG@10      21         76          -0.0034              0.0790     0.0824         664.0 0.242178     1.000000        False
 Q2D_Llama3 - Bo1      nDCG@10      21         76           0.0292              0.0906     0.0613         666.5 0.251182     1.000000        False
Q2D_Llama3 - BM25    AP(rel=2)      21         76           0.0613              0.1097     0.0485         626.0 0.131081     1.000000        False
 Q2D_Llama3 - RM3    AP(rel=2)      21         76           0.0359              0.0817     0.0457         639.0 0.165037     1.000000        False
 Q2D_Llama3 - Bo1    AP(rel=2)      21         76      

In [15]:
# This cell prints only the statistically significant difficulty effects so the main takeaway is easy to read.
print("=== Significant difficulty effects on Query2Doc gains ===")
sig_difficulty = difficulty_delta_tests[difficulty_delta_tests["significant"]].sort_values(["measure", "p_corrected"])
if len(sig_difficulty) == 0:
    print("No significant hard vs non-hard differences in Query2Doc gains after Holm correction.")
else:
    print(
        sig_difficulty[
            [
                "comparison",
                "measure",
                "n_hard",
                "n_nonhard",
                "hard_mean_delta",
                "nonhard_mean_delta",
                "delta_gap",
                "p_corrected",
            ]
        ].to_string(index=False)
    )


=== Significant difficulty effects on Query2Doc gains ===
No significant hard vs non-hard differences in Query2Doc gains after Holm correction.


In [16]:
# This cell tests whether Query2Doc's gains vary by query intent and answer type,
# then reports secondary within-group paired Q2D-vs-traditional comparisons.
from itertools import combinations
from scipy.stats import kruskal, mannwhitneyu, wilcoxon
from statsmodels.stats.multitest import multipletests

comparison_order = [f"{target_method} - {baseline}" for baseline in traditional_methods]
intent_order = [
    group
    for group in ["Informational", "Fact-Seeking", "Data/List"]
    if group in delta_results["meta_intent"].dropna().unique()
]
answer_type_order = [
    group
    for group in ["Narrative", "Atomic", "Complex"]
    if group in delta_results["meta_answer_type"].dropna().unique()
]

def apply_holm(df, p_col="p_value", group_cols=None):
    df = df.copy()
    if len(df) == 0:
        df["p_corrected"] = pd.Series(dtype=float)
        df["significant"] = pd.Series(dtype=bool)
        return df

    if group_cols:
        corrected_parts = []
        for _, part in df.groupby(group_cols, dropna=False, sort=False):
            reject, p_corr, _, _ = multipletests(part[p_col], method="holm")
            part = part.copy()
            part["p_corrected"] = p_corr
            part["significant"] = reject
            corrected_parts.append(part)
        return pd.concat(corrected_parts, ignore_index=True)

    reject, p_corr, _, _ = multipletests(df[p_col], method="holm")
    df["p_corrected"] = p_corr
    df["significant"] = reject
    return df

def run_delta_group_tests(delta_df, group_col, analysis_label, group_order):
    omnibus_rows = []
    pairwise_rows = []

    for comparison in comparison_order:
        for measure in metric_order:
            subset = delta_df[
                (delta_df["comparison"] == comparison)
                & (delta_df["measure"] == measure)
                & (delta_df[group_col].notna())
            ]
            group_scores = {}
            for group_name in group_order:
                scores = subset.loc[subset[group_col] == group_name, "delta"].dropna().values
                if len(scores) >= 2:
                    group_scores[group_name] = scores

            if len(group_scores) < 2:
                continue

            kw_stat, kw_p = kruskal(*[group_scores[group_name] for group_name in group_scores])
            omnibus_rows.append(
                {
                    "analysis": analysis_label,
                    "comparison": comparison,
                    "measure": measure,
                    "kw_statistic": round(float(kw_stat), 4),
                    "p_value": float(kw_p),
                    "group_means": {
                        group_name: round(float(group_scores[group_name].mean()), 4)
                        for group_name in group_scores
                    },
                }
            )

            if len(group_scores) >= 3:
                for group_1, group_2 in combinations(group_scores.keys(), 2):
                    scores_1 = group_scores[group_1]
                    scores_2 = group_scores[group_2]
                    stat, p = mannwhitneyu(scores_1, scores_2, alternative="two-sided")
                    pairwise_rows.append(
                        {
                            "analysis": analysis_label,
                            "comparison": comparison,
                            "measure": measure,
                            "group_1": group_1,
                            "group_2": group_2,
                            "n_group_1": len(scores_1),
                            "n_group_2": len(scores_2),
                            "mean_group_1": round(float(scores_1.mean()), 4),
                            "mean_group_2": round(float(scores_2.mean()), 4),
                            "delta_gap": round(float(scores_1.mean() - scores_2.mean()), 4),
                            "mw_statistic": round(float(stat), 4),
                            "p_value": float(p),
                        }
                    )

    omnibus_df = apply_holm(pd.DataFrame(omnibus_rows))
    pairwise_df = apply_holm(pd.DataFrame(pairwise_rows), group_cols=["analysis", "comparison", "measure"])

    if len(omnibus_df) > 0:
        omnibus_df["comparison"] = pd.Categorical(omnibus_df["comparison"], categories=comparison_order, ordered=True)
        omnibus_df["measure"] = pd.Categorical(omnibus_df["measure"], categories=metric_order, ordered=True)
    if len(pairwise_df) > 0:
        pairwise_df["comparison"] = pd.Categorical(pairwise_df["comparison"], categories=comparison_order, ordered=True)
        pairwise_df["measure"] = pd.Categorical(pairwise_df["measure"], categories=metric_order, ordered=True)
    return omnibus_df, pairwise_df

def run_within_group_q2d_tests(results_df, group_col, analysis_label, group_order):
    rows = []
    for group_name in group_order:
        group_df = results_df[results_df[group_col] == group_name]
        for measure in metric_order:
            score_pivot = (
                group_df[group_df["measure"] == measure]
                .pivot(index="qid", columns="name", values="value")
            )
            if target_method not in score_pivot.columns:
                continue

            for baseline_name in traditional_methods:
                if baseline_name not in score_pivot.columns:
                    continue
                paired = score_pivot[[target_method, baseline_name]].dropna()
                if len(paired) < 2:
                    continue

                diffs = paired[target_method] - paired[baseline_name]
                if (diffs == 0).all():
                    stat, p = 0.0, 1.0
                else:
                    stat, p = wilcoxon(paired[target_method], paired[baseline_name], alternative="two-sided")

                rows.append(
                    {
                        "analysis": analysis_label,
                        "group": group_name,
                        "comparison": f"{target_method} - {baseline_name}",
                        "measure": measure,
                        "n_pairs": len(paired),
                        "target_mean": round(float(paired[target_method].mean()), 4),
                        "baseline_mean": round(float(paired[baseline_name].mean()), 4),
                        "mean_delta": round(float(diffs.mean()), 4),
                        "median_delta": round(float(diffs.median()), 4),
                        "wilcoxon_stat": round(float(stat), 4),
                        "p_value": float(p),
                    }
                )

    within_df = apply_holm(pd.DataFrame(rows), group_cols=["analysis", "group"])
    if len(within_df) > 0:
        within_df["comparison"] = pd.Categorical(within_df["comparison"], categories=comparison_order, ordered=True)
        within_df["measure"] = pd.Categorical(within_df["measure"], categories=metric_order, ordered=True)
    return within_df

intent_omnibus, intent_pairwise = run_delta_group_tests(
    delta_results,
    "meta_intent",
    "Query Intent",
    intent_order,
)
print("=== Query2Doc gain differences by Query Intent ===")
print(intent_omnibus.sort_values(["measure", "p_corrected"]).to_string(index=False))
print("--- Significant pairwise intent differences only ---")
sig_intent_pairwise = intent_pairwise[intent_pairwise["significant"]].sort_values(["measure", "comparison", "p_corrected"])
if len(sig_intent_pairwise) == 0:
    print("No significant pairwise intent differences after Holm correction.")
else:
    print(
        sig_intent_pairwise[
            [
                "comparison",
                "measure",
                "group_1",
                "group_2",
                "n_group_1",
                "n_group_2",
                "mean_group_1",
                "mean_group_2",
                "delta_gap",
                "p_corrected",
            ]
        ].to_string(index=False)
    )

answer_omnibus, answer_pairwise = run_delta_group_tests(
    delta_results,
    "meta_answer_type",
    "Answer Type",
    answer_type_order,
)
print("\n=== Query2Doc gain differences by Answer Type ===")
print(answer_omnibus.sort_values(["measure", "p_corrected"]).to_string(index=False))
print("--- Significant pairwise answer-type differences only ---")
sig_answer_pairwise = answer_pairwise[answer_pairwise["significant"]].sort_values(["measure", "comparison", "p_corrected"])
if len(sig_answer_pairwise) == 0:
    print("No significant pairwise answer-type differences after Holm correction.")
else:
    print(
        sig_answer_pairwise[
            [
                "comparison",
                "measure",
                "group_1",
                "group_2",
                "n_group_1",
                "n_group_2",
                "mean_group_1",
                "mean_group_2",
                "delta_gap",
                "p_corrected",
            ]
        ].to_string(index=False)
    )

intent_within = run_within_group_q2d_tests(results, "meta_intent", "Query Intent", intent_order)
print("\n=== Secondary within-intent paired Query2Doc vs baseline tests ===")
sig_intent_within = intent_within[intent_within["significant"]].sort_values(["group", "measure", "p_corrected"])
if len(sig_intent_within) == 0:
    print("No significant within-intent Query2Doc vs baseline differences after Holm correction.")
else:
    print(sig_intent_within.to_string(index=False))

answer_within = run_within_group_q2d_tests(results, "meta_answer_type", "Answer Type", answer_type_order)
print("\n=== Secondary within-answer-type paired Query2Doc vs baseline tests ===")
sig_answer_within = answer_within[answer_within["significant"]].sort_values(["group", "measure", "p_corrected"])
if len(sig_answer_within) == 0:
    print("No significant within-answer-type Query2Doc vs baseline differences after Holm correction.")
else:
    print(sig_answer_within.to_string(index=False))


=== Query2Doc gain differences by Query Intent ===
    analysis        comparison      measure  kw_statistic  p_value                                                            group_means  p_corrected  significant
Query Intent Q2D_Llama3 - BM25      nDCG@10        0.6010 0.740460 {'Informational': 0.0927, 'Fact-Seeking': 0.0768, 'Data/List': 0.0975}     1.000000        False
Query Intent  Q2D_Llama3 - RM3      nDCG@10        0.0548 0.972984 {'Informational': 0.0642, 'Fact-Seeking': 0.0585, 'Data/List': 0.0548}     1.000000        False
Query Intent  Q2D_Llama3 - Bo1      nDCG@10        0.2371 0.888219  {'Informational': 0.0882, 'Fact-Seeking': 0.0783, 'Data/List': 0.044}     1.000000        False
Query Intent Q2D_Llama3 - BM25    AP(rel=2)        2.1478 0.341677 {'Informational': 0.1054, 'Fact-Seeking': 0.0552, 'Data/List': 0.1209}     1.000000        False
Query Intent  Q2D_Llama3 - RM3    AP(rel=2)        2.3663 0.306308 {'Informational': 0.0756, 'Fact-Seeking': 0.0418, 'Data/List':